<a href="https://colab.research.google.com/gist/chottokun/e7dfdb5a74ff746085b8ec28e56fb512/unsloth-inkling-2-mini-q4km-eval-cuda13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦥 Unsloth Inkling-2-Mini (UD-Q4_K_M + DSpark + Vision) 総合性能評価＆量子化比較ノートブック

**概要**:
`unsloth/Inkling-2-Mini-GGUF` の **`UD-Q4_K_M` (16.61GB, Top-1 一致率 97.9%, Mean KLD 0.0108)** を評価し、先行して実行した **`UD-Q4_K_XL` (17.63GB, Top-1 一致率 98.0%, Mean KLD 0.0107)** および過去モデル（Qwen3.8-27B / Gemma-4-31B / Gemma-4-E4B）と徹底比較するための総合性能評価ノートブックです。

---

### 🌟 `UD-Q4_K_M` vs `UD-Q4_K_XL` 比較検証のポイント
1. **約 1.02 GB の VRAM 節約効果と KV キャッシュ余裕**:
   - `UD-Q4_K_XL` (17.63GB) に対して `UD-Q4_K_M` (16.61GB) はモデル重みが **1.02 GB 軽量** であり、NVIDIA L4 (24GB VRAM) 上で `mmproj-F16` (0.64GB) + `DSpark Q8_0` (0.40GB) を同時ロードした際、長文コンテキスト（KV キャッシュ）により多くの VRAM を割り当て可能です。
2. **量子化劣化の有無を同一母集団 (`SEED=42`) で厳密検証**:
   - Unsloth 公式ベンチマークでは `UD-Q4_K_XL` と `UD-Q4_K_M` の KLD 差はわずか `0.0001`（`0.0107` vs `0.0108`）です。
   - 実際の **WikiText-2 Perplexity**、**MMLU (285問)**、**ARC-Challenge (200問)**、**GSM8K (100問)**、**DSpark 推論速度**、**Vision チャート読解**、**日本語 IFEval 三重制約** において、1GB の軽量化が精度や生成速度にどう影響するかを自動比較します。
3. **Google Drive 上の `UD-Q4_K_XL` 結果と自動突合**:
   - `/content/drive/MyDrive/unsloth-inkling-2-mini-eval-results/results.json`（`UD-Q4_K_XL` の評価結果）が存在する場合、自動的に読み込んで **`UD-Q4_K_XL` vs `UD-Q4_K_M` 直接対決比較セクション** をレポート (`detailed_evaluation_report.md`) に出力します。

## 1. GPU 環境確認

In [1]:
!nvidia-smi

Sun Nov 22 01:23:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   53C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Google Drive のマウント & 結果保存先フォルダの準備

In [2]:
import os
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError as e:
    if "already contain files" in str(e) or "already mounted" in str(e):
        print("✅ Google Drive はすでにマウントされています。")
    else:
        raise e

DRIVE_OUT = '/content/drive/MyDrive/unsloth-inkling-2-mini-q4km-eval-results'
LOCAL_OUT = '/content/results'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(LOCAL_OUT, exist_ok=True)
print(f"✅ 結果保存先フォルダ (Google Drive): {DRIVE_OUT}")
print(f"✅ 結果保存先フォルダ (ローカル): {LOCAL_OUT}")

Mounted at /content/drive
✅ 結果保存先フォルダ (Google Drive): /content/drive/MyDrive/unsloth-inkling-2-mini-q4km-eval-results
✅ 結果保存先フォルダ (ローカル): /content/results


## 3. 依存ライブラリ導入 & Unsloth 専用 `llama.cpp (b12588-mix-c0c3de8)` CUDA 自動適合セットアップ

> [!IMPORTANT]
> **GPU オフロード必須対策**:
> Google Colab のランタイムは近年 **CUDA 13.0 (`libcudart.so.13`, `libcublas.so.13`)** に更新されています。そのため、`cuda12-portable` バイナリをそのまま配置すると動的リンク時に `libggml-cuda.so` のロードに失敗し、サイレントに CPU フォールバック（約10 tok/s）してしまいます。
> 以下のセルでは、システムの CUDA メジャーバージョン（CUDA 13 または CUDA 12）を自動判定して最適な Unsloth プリビルトバイナリを導入し、さらに **`llama-bench --list-devices` で `NVIDIA L4 / A100 / T4` が CUDA バックエンドとして認識されていることを強制検証**します（万一プリビルトの動的リンクが不一致の場合は、自動的に Unsloth ソースコード `llama.cpp-source-b12588-mix-c0c3de8.tar.gz` からネイティブ CUDA ビルドを行います）。

In [3]:
%%bash
set -e
# Colab 標準の pandas / requests 等を壊さないよう、不足パッケージのみ導入
pip install -q huggingface_hub hf_transfer datasets tabulate

LLAMA_DIR="/content/llama.cpp"
rm -rf "${LLAMA_DIR}"
mkdir -p "${LLAMA_DIR}"

# システムの CUDA バージョン (12 or 13) を自動検出
CUDA_MAJOR="13"
if [ -e /usr/local/cuda/lib64/libcudart.so.13 ] || ldconfig -p | grep -q "libcudart.so.13"; then
    CUDA_MAJOR="13"
elif [ -e /usr/local/cuda/lib64/libcudart.so.12 ] || ldconfig -p | grep -q "libcudart.so.12"; then
    CUDA_MAJOR="12"
fi

if [ "${CUDA_MAJOR}" = "13" ]; then
    ARCHIVE_NAME="app-b12588-mix-c0c3de8-linux-x64-cuda13-older.tar.gz"
else
    ARCHIVE_NAME="app-b12588-mix-c0c3de8-linux-x64-cuda12-older.tar.gz"
fi

echo "=== Unsloth llama.cpp (${ARCHIVE_NAME} for CUDA ${CUDA_MAJOR}) をダウンロード・展開中 ==="
curl -L --retry 3 -o /content/llama.tar.gz \
    "https://github.com/unslothai/llama.cpp/releases/download/b12588-mix-c0c3de8/${ARCHIVE_NAME}"
tar -xzf /content/llama.tar.gz -C "${LLAMA_DIR}"
rm -f /content/llama.tar.gz
chmod +x "${LLAMA_DIR}"/llama-*

export LD_LIBRARY_PATH="${LLAMA_DIR}:/usr/local/cuda/lib64:${LD_LIBRARY_PATH}"

echo "=== CUDA バックエンド認識チェック ==="
if ! ldd "${LLAMA_DIR}/libggml-cuda.so" | grep -q "not found" && "${LLAMA_DIR}/llama-server" --list-devices 2>&1 | grep -qiE "CUDA|NVIDIA|Tesla"; then
    echo "✅ プリビルト CUDA バイナリで GPU が正常に認識されました！"
    "${LLAMA_DIR}/llama-server" --list-devices
else
    echo "⚠️ プリビルトバイナリの共有ライブラリ不一致を検出。Unsloth 公式ソース (b12588-mix-c0c3de8) から CUDA ネイティブビルドを実行します..."
    ldd "${LLAMA_DIR}/libggml-cuda.so" || true
    rm -rf "${LLAMA_DIR}" /content/unsloth_src
    mkdir -p /content/unsloth_src "${LLAMA_DIR}"
    curl -L --retry 3 -o /content/llama_src.tar.gz \
        "https://github.com/unslothai/llama.cpp/releases/download/b12588-mix-c0c3de8/llama.cpp-source-b12588-mix-c0c3de8.tar.gz"
    tar -xzf /content/llama_src.tar.gz -C /content/unsloth_src --strip-components=1
    cd /content/unsloth_src
    cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF
    cmake --build build -j$(nproc) --target llama-server llama-bench llama-perplexity
    cp -a build/bin/* "${LLAMA_DIR}/"
    find build -name "*.so*" -exec cp -a {} "${LLAMA_DIR}/" \;
    echo "✅ ソースからの CUDA ビルド完了！"
    "${LLAMA_DIR}/llama-server" --list-devices
fi

"${LLAMA_DIR}/llama-server" --version

=== Unsloth llama.cpp (app-b12588-mix-c0c3de8-linux-x64-cuda13-older.tar.gz for CUDA 13) をダウンロード・展開中 ===
=== CUDA バックエンド認識チェック ===
✅ プリビルト CUDA バイナリで GPU が正常に認識されました！
Available devices:
  CUDA0: NVIDIA L4 (22563 MiB, 22369 MiB free)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  235M  100  235M    0     0  33.1M      0  0:00:07  0:00:07 --:--:-- 39.2M
version: 1.0.0-dev (build 12588, commit 883286076)
built with GNU 11.4.0 for Linux x86_64 (Compiled by the Unsloth team)


## 4. Inkling-2-Mini GGUF (`UD-Q4_K_M`: 16.61GB) + Vision Projector (`mmproj-F16`) + DSpark Drafter の高速ダウンロード

本ノートブックでは VRAM 効率重視の **`Inkling-2-Mini-UD-Q4_K_M.gguf` (16.61GB)** をダウンロードします（16GB クラスの T4 環境で実行された場合は自動的に `Inkling-2-Mini-UD-IQ3_XXS.gguf` (11.66GB) にフォールバックします）。

In [4]:
import os
import time
import subprocess
from huggingface_hub import hf_hub_download

MODEL_REPO = "unsloth/Inkling-2-Mini-GGUF"
LOCAL_DIR = "/content/models"
os.makedirs(LOCAL_DIR, exist_ok=True)

# VRAM 容量を検出して最適な量子化モデルを選択 (L4/A100 -> UD-Q4_K_XL, T4 -> UD-IQ3_XXS)
vram_mb = 24000
try:
    out = subprocess.run(
        "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits",
        shell=True, capture_output=True, text=True
    ).stdout.strip().splitlines()[0]
    vram_mb = int(out)
except Exception as e:
    print(f"[!] VRAM 自動検出スキップ ({e}) - デフォルトで UD-Q4_K_M を使用します。")

if vram_mb >= 20000:
    MAIN_GGUF = "Inkling-2-Mini-UD-Q4_K_M.gguf"
    QUANT_NAME = "UD-Q4_K_M (Dynamic 3.0, 16.61GB)"
    MIN_BYTES = 15 * (1024**3)
else:
    print(f"⚠️ VRAM {vram_mb} MiB (< 20GB) を検出したため、T4 互換の UD-IQ3_XXS (11.66GB) に自動フォールバックします。")
    MAIN_GGUF = "Inkling-2-Mini-UD-IQ3_XXS.gguf"
    QUANT_NAME = "UD-IQ3_XXS (Dynamic 3.0, 11.66GB - T4 Fallback)"
    MIN_BYTES = 10 * (1024**3)

MMPROJ_GGUF = "mmproj-F16.gguf"
DSPARK_GGUF = "dspark-Inkling-2-Mini-Q8_0.gguf"

FILES_TO_DOWNLOAD = [
    (MAIN_GGUF, MIN_BYTES),
    (MMPROJ_GGUF, 500 * (1024**2)),
    (DSPARK_GGUF, 300 * (1024**2)),
]

def download_file_resilient(repo_id, filename, min_size):
    target_path = os.path.join(LOCAL_DIR, filename)
    if os.path.exists(target_path) and os.path.getsize(target_path) >= min_size:
        size_gb = os.path.getsize(target_path) / (1024**3)
        print(f"✅ 既にダウンロード済み: {filename} ({size_gb:.2f} GB)")
        return target_path

    print(f"=== {filename} をダウンロード中 ===")
    success = False
    for attempt in range(3):
        try:
            print(f"[*] 試行 {attempt + 1}/3 (HF Transfer 有効)...")
            os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
            hf_hub_download(repo_id=repo_id, filename=filename, local_dir=LOCAL_DIR)
            success = True
            break
        except Exception as e:
            print(f"[!] HF Transfer エラー: {e}")
            time.sleep(5)

    if not success:
        print("[*] 標準 HTTP ダウンロードへフォールバック中...")
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
        for attempt in range(3):
            try:
                hf_hub_download(repo_id=repo_id, filename=filename, local_dir=LOCAL_DIR)
                success = True
                break
            except Exception as e:
                print(f"[!] 標準ダウンロードエラー: {e}")
                time.sleep(10)

    if not success:
        subprocess.run(f"huggingface-cli download {repo_id} {filename} --local-dir {LOCAL_DIR}", shell=True, check=True)

    size_gb = os.path.getsize(target_path) / (1024**3)
    print(f"✅ ダウンロード完了: {target_path} ({size_gb:.2f} GB)")
    return target_path

for fname, msize in FILES_TO_DOWNLOAD:
    download_file_resilient(MODEL_REPO, fname, msize)

with open("/content/models/selected_model.json", "w") as f:
    import json
    json.dump({"main_gguf": MAIN_GGUF, "quant_name": QUANT_NAME, "mmproj": MMPROJ_GGUF, "dspark": DSPARK_GGUF}, f)

=== Inkling-2-Mini-UD-Q4_K_M.gguf をダウンロード中 ===
[*] 試行 1/3 (HF Transfer 有効)...


Inkling-2-Mini-UD-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 16.6GB            

Inkling-2-Mini-UD-Q4_K_M.gguf: downloading bytes:           |  0.00B            

✅ ダウンロード完了: /content/models/Inkling-2-Mini-UD-Q4_K_M.gguf (15.47 GB)
=== mmproj-F16.gguf をダウンロード中 ===
[*] 試行 1/3 (HF Transfer 有効)...


mmproj-F16.gguf: reconstructing file:   0%|          |  0.00B /  639MB            

mmproj-F16.gguf: downloading bytes:           |  0.00B            

✅ ダウンロード完了: /content/models/mmproj-F16.gguf (0.60 GB)
=== dspark-Inkling-2-Mini-Q8_0.gguf をダウンロード中 ===
[*] 試行 1/3 (HF Transfer 有効)...


dspark-Inkling-2-Mini-Q8_0.gguf: reconstructing file:   0%|          |  0.00B /  402MB            

dspark-Inkling-2-Mini-Q8_0.gguf: downloading bytes:           |  0.00B            

✅ ダウンロード完了: /content/models/dspark-Inkling-2-Mini-Q8_0.gguf (0.37 GB)


## 5. 総合評価スイートの実行（定量585問・Perplexity・DSpark加速検証・Vision評価・Tool Calling・過去モデル横断比較レポート生成）

In [5]:
import base64
import json
import os
import random
import re
import subprocess
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from datasets import load_dataset

with open("/content/models/selected_model.json") as f:
    sel = json.load(f)

MODEL = f"/content/models/{sel['main_gguf']}"
MMPROJ = f"/content/models/{sel['mmproj']}"
DSPARK = f"/content/models/{sel['dspark']}"
QUANT_NAME = sel["quant_name"]

BIN = "/content/llama.cpp"
LOCAL_OUT = "/content/results"
DRIVE_OUT = "/content/drive/MyDrive/unsloth-inkling-2-mini-q4km-eval-results"
Q4KXL_RESULTS_PATH = "/content/drive/MyDrive/unsloth-inkling-2-mini-eval-results/results.json"
PORT = 8088
server_log_path = f"{LOCAL_OUT}/server.log"

os.makedirs(LOCAL_OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# 共有ライブラリパスを設定
os.environ["LD_LIBRARY_PATH"] = f"{BIN}:/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
dev_check = subprocess.run(f"{BIN}/llama-server --list-devices", shell=True, capture_output=True, text=True, env=os.environ)
print("=== llama.cpp 認識デバイス一覧 ===\n" + (dev_check.stdout + dev_check.stderr).strip())
if "CUDA" not in (dev_check.stdout + dev_check.stderr) and "NVIDIA" not in (dev_check.stdout + dev_check.stderr):
    raise RuntimeError("❌ GPU (CUDA バックエンド) が llama.cpp に認識されていません！セクション 3 を再実行してください。")

SEED = 42
results = {
    "model": "unsloth/Inkling-2-Mini-GGUF",
    "base_model": "thinkingmachines/Inkling-2-Mini",
    "architecture": "Looped Transformer MoE (24 layers, middle 12 layers looped 3x = 48 layer apps; 26.2B total / 3.4B active / 5.4B effective active)",
    "quant": QUANT_NAME,
    "started": time.strftime("%Y-%m-%d %H:%M:%S"),
    "tasks": {}
}

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def save_all():
    """各タスク完了ごとに Drive とローカルの双方へ結果を退避"""
    for d in [LOCAL_OUT, DRIVE_OUT]:
        with open(os.path.join(d, "results.json"), "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

# GPU 情報を記録
gpu_info = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
    shell=True, capture_output=True, text=True
).stdout.strip()
results["gpu"] = gpu_info
log(f"Target GPU: {results['gpu']} | Model: {MODEL}")
save_all()

# ================================================================
# 1. llama-bench (Throughput: Prompt Processing & Generation)
# ================================================================
log("=== [1/9] llama-bench (推論スループット計測) ===")
r = subprocess.run(
    f"{BIN}/llama-bench -m {MODEL} -ngl 99 -p 512,1024,2048,4096 -n 128,256 -r 3 -o json",
    shell=True, capture_output=True, text=True, env=os.environ
)
try:
    bench = json.loads(r.stdout)
    results["throughput"] = [
        {
            "test": f"Prompt {b['n_prompt']} tok" if b["n_prompt"] else f"Generation {b['n_gen']} tok",
            "tokens_per_sec": round(b["avg_ts"], 2),
            "stddev": round(b["stddev_ts"], 2)
        }
        for b in bench
    ]
except Exception as e:
    results["throughput"] = {"error": str(e), "raw": (r.stdout + r.stderr)[:800]}
log(f"Throughput: {results['throughput']}")
save_all()

# ================================================================
# 2. llama-perplexity (WikiText-2 言語モデリング精度)
# ================================================================
log("=== [2/9] llama-perplexity (WikiText-2 言語モデリング精度) ===")
try:
    wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    with open("/content/wiki.test.raw", "w", encoding="utf-8") as f:
        f.write("\n".join(wt["text"]))

    r = subprocess.run(
        f"{BIN}/llama-perplexity -m {MODEL} -ngl 99 -c 2048 --chunks 40 -f /content/wiki.test.raw",
        shell=True, capture_output=True, text=True, env=os.environ
    )
    m = re.search(r"Final estimate: PPL = ([\d.]+) \+/- ([\d.]+)", r.stdout + r.stderr)
    if m:
        results["perplexity_wikitext2"] = {
            "ppl": float(m.group(1)),
            "stderr": float(m.group(2)),
            "ctx": 2048,
            "chunks": 40
        }
    else:
        results["perplexity_wikitext2"] = {"raw": (r.stdout + r.stderr)[-800:]}
except Exception as e:
    results["perplexity_wikitext2"] = {"error": str(e)}
log(f"Perplexity: {results['perplexity_wikitext2']}")
save_all()

# ================================================================
# サーバー起動・チャット共通ヘルパー
# ================================================================
def start_llama_server(use_dspark=True, use_mmproj=True, ctx_size=8192):
    cmd = [
        f"{BIN}/llama-server",
        "--model", MODEL,
        "--jinja",
        "--ctx-size", str(ctx_size),
        "--n-gpu-layers", "99",
        "--parallel", "1",
        "--temp", "1.0",
        "--top-p", "1.0",
        "--top-k", "0",
        "--min-p", "0.0",
        "--host", "127.0.0.1",
        "--port", str(PORT)
    ]
    if use_mmproj and os.path.exists(MMPROJ):
        cmd.extend(["--mmproj", MMPROJ])
    if use_dspark and os.path.exists(DSPARK):
        cmd.extend([
            "--model-draft", DSPARK,
            "--spec-type", "draft-dspark",
            "--spec-draft-n-max", "3",
            "--n-gpu-layers-draft", "99"
        ])
    cmd_str = " ".join(cmd) + f" > {server_log_path} 2>&1"
    proc = subprocess.Popen(cmd_str, shell=True, env=os.environ)
    for _ in range(180):
        try:
            req = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3)
            if json.load(req).get("status") == "ok":
                return proc
        except Exception:
            pass
        time.sleep(2)
    if os.path.exists(server_log_path):
        with open(server_log_path, "r", encoding="utf-8", errors="replace") as f:
            print(f.read()[-3000:])
    proc.terminate()
    raise RuntimeError("llama-server の起動に失敗しました。")

def chat(messages_or_prompt, max_tokens=512, temperature=0.0, tools=None):
    if isinstance(messages_or_prompt, str):
        messages = [{"role": "user", "content": messages_or_prompt}]
    else:
        messages = messages_or_prompt

    payload = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": 1.0,
        "top_k": 0,
        "min_p": 0.0
    }
    if tools:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"

    body = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        data=body,
        headers={"Content-Type": "application/json"}
    )
    t0 = time.time()
    resp = json.load(urllib.request.urlopen(req, timeout=600))
    dt = time.time() - t0
    choice = resp["choices"][0]
    msg = choice["message"]
    content = msg.get("content") or ""
    reasoning = msg.get("reasoning_content") or ""
    full_text = f"<thought>\n{reasoning}\n</thought>\n{content}".strip() if (reasoning and content) else (content or reasoning)
    tool_calls = msg.get("tool_calls")
    timings = resp.get("timings", {})
    return full_text, content, reasoning, choice.get("finish_reason"), resp.get("usage", {}), dt, tool_calls, timings

# ================================================================
# 3. DSpark Speculative Decoding 加速効果ベンチマーク (OFF vs ON)
# ================================================================
log("=== [3/9] DSpark Speculative Decoding (投機的デコーディング) 加速効果計測 ===")
dspark_prompts = [
    {"name": "Python MergeSort & QuickSort Implementation", "prompt": "Write a complete, well-commented Python implementation of both MergeSort and QuickSort, including complexity analysis and unit tests."},
    {"name": "Distributed Consensus (Raft vs Paxos)", "prompt": "Explain the architectural differences between Raft and Paxos consensus algorithms in detail, focusing on leader election, log replication, and safety guarantees."},
    {"name": "Mathematical Derivation (Euler-Lagrange)", "prompt": "Derive the Euler-Lagrange equation from the principle of stationary action step by step, and apply it to find the equations of motion for a simple harmonic oscillator."}
]

dspark_comparison = {"without_dspark": [], "with_dspark": []}

# 3A: DSpark OFF で計測
log("[*] 3A: DSpark OFF (ベースライン) で llama-server を起動中...")
srv_no_draft = start_llama_server(use_dspark=False, use_mmproj=False, ctx_size=8192)
for dp in dspark_prompts:
    full_t, _, _, fin, usage, dt, _, timings = chat(dp["prompt"], max_tokens=512, temperature=0.0)
    comp_toks = usage.get("completion_tokens", 0)
    tps = round(comp_toks / max(dt, 0.001), 2)
    dspark_comparison["without_dspark"].append({
        "task": dp["name"],
        "completion_tokens": comp_toks,
        "elapsed_sec": round(dt, 2),
        "tokens_per_sec": tps,
        "timings": timings
    })
    log(f"  [DSpark OFF] {dp['name']}: {tps} tok/s ({comp_toks} toks in {dt:.2f}s)")
srv_no_draft.terminate()
srv_no_draft.wait()
time.sleep(3)

# 3B: DSpark ON (+ mmproj-F16) で起動（以降の評価でも継続使用）
log("[*] 3B: DSpark ON (draft-dspark, n-max=3) + Vision Projector (mmproj-F16) で llama-server を起動中...")
server = start_llama_server(use_dspark=True, use_mmproj=True, ctx_size=8192)
log(f"✅ llama-server (DSpark + Vision 有効) がポート {PORT} で正常に起動しました。")

for dp in dspark_prompts:
    full_t, _, _, fin, usage, dt, _, timings = chat(dp["prompt"], max_tokens=512, temperature=0.0)
    comp_toks = usage.get("completion_tokens", 0)
    tps = round(comp_toks / max(dt, 0.001), 2)
    dspark_comparison["with_dspark"].append({
        "task": dp["name"],
        "completion_tokens": comp_toks,
        "elapsed_sec": round(dt, 2),
        "tokens_per_sec": tps,
        "timings": timings
    })
    log(f"  [DSpark ON]  {dp['name']}: {tps} tok/s ({comp_toks} toks in {dt:.2f}s)")

avg_off = sum(x["tokens_per_sec"] for x in dspark_comparison["without_dspark"]) / len(dspark_prompts)
avg_on = sum(x["tokens_per_sec"] for x in dspark_comparison["with_dspark"]) / len(dspark_prompts)
dspark_comparison["avg_tok_per_sec_off"] = round(avg_off, 2)
dspark_comparison["avg_tok_per_sec_on"] = round(avg_on, 2)
dspark_comparison["speedup_ratio"] = round(avg_on / max(avg_off, 0.01), 2)
results["dspark_benchmark"] = dspark_comparison
log(f"⚡ DSpark 平均速度比較: OFF={avg_off:.2f} tok/s -> ON={avg_on:.2f} tok/s (倍率: {dspark_comparison['speedup_ratio']}x)")
save_all()

# ================================================================
# ベンチマーク共通ドライバ
# ================================================================
def run_bench(name, items, build_prompt, parse_fn, max_tokens=512):
    log(f"実行中: {name} (計 {len(items)} 問, max_tokens={max_tokens})")
    t0 = time.time()
    def process_item(it):
        try:
            full_t, content_t, reasoning_t, fin, usage, dt, _, _ = chat(build_prompt(it), max_tokens=max_tokens, temperature=0.0)
        except Exception as e:
            full_t, content_t, reasoning_t, fin, usage, dt = f"ERROR: {e}", "", "", "error", {}, 0
            print(f"Chat Error on {name}: {e}")
        # 最終回答テキスト（content）を優先してパースし、なければ全体からパース
        pred = parse_fn(content_t) if content_t else None
        if pred is None:
            pred = parse_fn(full_t)
        return {
            "gold": it.get("gold"),
            "pred": pred,
            "correct": (pred == it.get("gold")),
            "finish": fin,
            "completion_tokens": usage.get("completion_tokens", 0),
            "response": full_t
        }
    # --parallel 1 構成のため 1 ワーカーで順番かつ高速に処理
    with ThreadPoolExecutor(max_workers=1) as ex:
        rows = list(ex.map(process_item, items))

    for d in [LOCAL_OUT, DRIVE_OUT]:
        with open(os.path.join(d, f"{name}_details.jsonl"), "w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    acc = sum(r["correct"] for r in rows) / len(rows) if rows else 0
    results[name] = {
        "n": len(rows),
        "accuracy": round(acc, 4),
        "truncated": sum(r["finish"] == "length" for r in rows),
        "errors": sum(r["finish"] == "error" for r in rows),
        "avg_tokens": round(sum(r.get("completion_tokens", 0) for r in rows) / max(1, len(rows)), 1),
        "elapsed_sec": round(time.time() - t0, 1)
    }
    log(f"✅ {name} 完了: 正答率 {acc*100:.2f}% ({len(rows)}問, {results[name]['elapsed_sec']}秒, 切断:{results[name]['truncated']}件)")
    save_all()

LET = "ABCD"

def parse_letter(t):
    if not t:
        return None
    t_clean = re.sub(r"<thought>.*?</thought>", "", t, flags=re.S)
    t_clean = re.sub(r"<[^>]+>", "", t_clean)
    # 末尾の Answer: X を優先検索
    matches = re.findall(r"(?:answer|答え|正解)\s*(?:is|:)?\s*\**\(?([A-D])\)?\**\b", t_clean, re.I)
    if matches:
        return matches[-1].upper()
    lines = [l.strip() for l in t_clean.strip().split("\n") if l.strip()]
    if lines:
        m = re.search(r"\b([A-D])\b", lines[-1])
        if m:
            return m.group(1).upper()
    all_lets = re.findall(r"\b([A-D])\b", t_clean)
    return all_lets[-1].upper() if all_lets else None

def mc_prompt(it):
    opts = "\n".join(f"{LET[i]}. {c}" for i, c in enumerate(it["choices"]))
    return f"{it['q']}\n\n{opts}\n\nThink briefly (2-3 sentences max), then finish with the exact line 'Answer: <letter>' (where <letter> is A, B, C, or D)."

# ================================================================
# 4. MMLU (57分野 × 各5問 = 285問, SEED=42)
# ================================================================
log("=== [4/9] MMLU (57 分野総合知識ベンチマーク 285問) 準備中 ===")
mmlu = load_dataset("cais/mmlu", "all", split="test")
by_subj = {}
for row in mmlu:
    by_subj.setdefault(row["subject"], []).append(row)
rng = random.Random(SEED)
mmlu_items = []
for s in sorted(by_subj):
    for row in rng.sample(by_subj[s], 5):
        mmlu_items.append({"q": row["question"], "choices": row["choices"], "gold": LET[row["answer"]][0]})
run_bench("mmlu", mmlu_items, mc_prompt, parse_letter, max_tokens=512)

# ================================================================
# 5. ARC-Challenge (200問, SEED=42)
# ================================================================
log("=== [5/9] ARC-Challenge (難関科学推論 200問) 準備中 ===")
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
arc_rows = [r for r in arc if len(r["choices"]["label"]) == 4]
arc_items = []
for row in random.Random(SEED).sample(arc_rows, 200):
    labels = row["choices"]["label"]
    arc_items.append({
        "q": row["question"],
        "choices": row["choices"]["text"],
        "gold": LET[labels.index(row["answerKey"])]
    })
run_bench("arc_challenge", arc_items, mc_prompt, parse_letter, max_tokens=512)

# ================================================================
# 6. GSM8K (100問, SEED=42)
# ================================================================
log("=== [6/9] GSM8K (多段階数学推論 100問) 準備中 ===")
gsm = load_dataset("openai/gsm8k", "main", split="test")
gsm_items = [
    {"q": r["question"], "gold": r["answer"].split("####")[-1].strip().replace(",", "")}
    for r in random.Random(SEED).sample(list(gsm), 100)
]
def parse_num(t):
    if not t:
        return None
    t_clean = t.replace(",", "")
    matches = re.findall(r"Answer:\s*\**\$?(-?[\d.]+)", t_clean, re.I)
    nums = [matches[-1]] if matches else re.findall(r"-?\d+(?:\.\d+)?", t_clean)
    if not nums:
        return None
    n = nums[-1].rstrip(".")
    try:
        f = float(n)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return None

run_bench(
    "gsm8k", gsm_items,
    lambda it: f"{it['q']}\n\nSolve step by step concisely, then finish with a final line 'Answer: <number>'.",
    parse_num, max_tokens=1024
)

# ================================================================
# 7. マルチモーダル Vision 評価 (mmproj-F16: チャート読解 & 図面構造認識)
# ================================================================
log("=== [7/9] マルチモーダル Vision 評価 (チャート解析・視覚推論) ===")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

chart_path = f"{LOCAL_OUT}/eval_chart.png"
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
quarters = ["Q1 2026", "Q2 2026", "Q3 2026", "Q4 2026"]
rev_cloud = [120, 150, 185, 240]
rev_edge = [80, 95, 115, 160]
x = range(len(quarters))
w = 0.35
b1 = ax.bar([i - w/2 for i in x], rev_cloud, width=w, label="Cloud AI ($M)", color="#2563eb")
b2 = ax.bar([i + w/2 for i in x], rev_edge, width=w, label="Edge Looped AI ($M)", color="#16a34a")
ax.set_xticks(list(x))
ax.set_xticklabels(quarters)
ax.set_ylabel("Revenue (Million USD)")
ax.set_title("2026 Quarterly AI Infrastructure Revenue")
ax.legend()
for b in list(b1) + list(b2):
    h = b.get_height()
    ax.annotate(f"{h}", xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(chart_path)
plt.close()

with open(chart_path, "rb") as f:
    chart_b64 = base64.b64encode(f.read()).decode("utf-8")

vision_prompt = (
    "Analyze this bar chart carefully and answer the following questions:\n"
    "1. What is the exact revenue for both 'Cloud AI' and 'Edge Looped AI' in Q4 2026?\n"
    "2. What is the total combined revenue across all four quarters of 2026 for 'Edge Looped AI'?\n"
    "3. Which segment had the higher percentage growth from Q1 2026 to Q4 2026? Show the calculation."
)
vision_messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": vision_prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{chart_b64}"}}
        ]
    }
]
try:
    v_full, v_content, v_reas, v_fin, v_usage, v_dt, _, _ = chat(vision_messages, max_tokens=1024, temperature=0.0)
    # 正解検証: Q4 Cloud=240, Q4 Edge=160, Edge合計=80+95+115+160=450, 成長率両方+100% (120->240, 80->160)
    checks = {
        "q4_cloud_240": "240" in v_full,
        "q4_edge_160": "160" in v_full,
        "edge_total_450": "450" in v_full,
        "growth_100_percent": ("100%" in v_full or "double" in v_full.lower() or "2x" in v_full.lower())
    }
    results["vision_eval"] = {
        "prompt": vision_prompt,
        "checks": checks,
        "passed_checks": sum(checks.values()),
        "total_checks": len(checks),
        "completion_tokens": v_usage.get("completion_tokens", 0),
        "elapsed_sec": round(v_dt, 2),
        "response": v_full
    }
    log(f"✅ Vision 評価完了: チェック合格 {sum(checks.values())}/{len(checks)} ({v_dt:.2f}s)")
except Exception as e:
    results["vision_eval"] = {"error": str(e)}
    log(f"[!] Vision 評価エラー: {e}")
save_all()

# ================================================================
# 8. エージェント Tool Calling 整合性 & Looped Transformer トークン効率テスト
# ================================================================
log("=== [8/9] エージェント Tool Calling & Looped Transformer 思考トークン効率テスト ===")
tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_database_query",
            "description": "データベースでSQLを実行し結果を返します。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "実行するSQLクエリ"},
                    "limit": {"type": "integer", "description": "取得上限件数"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "bash_command",
            "description": "Linux ターミナルで指定したコマンドを実行します。",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "実行するシェルコマンド"}
                },
                "required": ["command"]
            }
        }
    }
]

tool_test_cases = [
    {"q": "users テーブルから status が 'active' かつ 2026年以降に登録された上位5名の名前とメールアドレスを取得するクエリを発行してください。", "func": "execute_database_query", "arg": "query"},
    {"q": "カレントディレクトリにある拡張子が .log のファイルのうち、容量が 10MB を超えるものを一覧表示するコマンドを実行してください。", "func": "bash_command", "arg": "command"}
]
tool_results = []
for tc in tool_test_cases:
    full_t, content_t, reas_t, fin, usage, dt, tcalls, _ = chat(tc["q"], max_tokens=512, temperature=0.0, tools=tools)
    success = False
    parsed = {}
    if tcalls:
        for c in tcalls:
            f_name = c.get("function", {}).get("name")
            raw_args = c.get("function", {}).get("arguments", "{}")
            try:
                parsed = json.loads(raw_args) if isinstance(raw_args, str) else raw_args
                if f_name == tc["func"] and tc["arg"] in parsed:
                    success = True
            except Exception:
                parsed = {"raw": raw_args}
    tool_results.append({
        "query": tc["q"],
        "expected_func": tc["func"],
        "success": success,
        "tool_calls": tcalls,
        "parsed_args": parsed,
        "response_text": full_t,
        "sec": round(dt, 2)
    })
results["agentic_tool_calling"] = tool_results
for d in [LOCAL_OUT, DRIVE_OUT]:
    with open(os.path.join(d, "tool_calling_details.jsonl"), "w", encoding="utf-8") as f:
        for r in tool_results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# トークン効率テスト (Token Efficiency & Looped Reasoning Check)
prompts_eff = [
    {"id": 1, "prompt": "ある水槽に給水管Aと排水管Bがある。Aだけで満水にするのに6時間かかり、満水からBだけで空にするのに9時間かかる。水槽が空の状態からAとBを同時に開いた場合、何時間で満水になるか？ステップを追って答えを導出してください。"},
    {"id": 2, "prompt": "3人の客がホテルに泊まり、1人10ドルずつ計30ドル支払った。支配人が5ドル値引きすることにしボーイに返却を頼んだが、ボーイは2ドルくすねて各人に1ドルずつ返した。結果3人は各9ドル計27ドル払い、ボーイが2ドル持っているので計29ドル。消えた1ドルはどうなったか論理的に矛盾を指摘せよ。"}
]
results["token_efficiency"] = []
for pe in prompts_eff:
    full_t, content_t, reas_t, fin, usage, dt, _, _ = chat(pe["prompt"], max_tokens=2048, temperature=1.0)
    toks = usage.get("completion_tokens", 0)
    results["token_efficiency"].append({
        "id": pe["id"],
        "prompt": pe["prompt"],
        "tokens": toks,
        "sec": round(dt, 2),
        "tok_per_sec": round(toks / max(dt, 0.001), 2),
        "output": full_t
    })
save_all()

# ================================================================
# 9. 日本語定性評価・厳格指示追従 (IFEval 三重制約)
# ================================================================
log("=== [9/9] 日本語定性評価 & 厳格指示追従 (IFEval 三重制約) ===")
ja_prompts = [
    {
        "type": "厳格指示追従 (IFEval形式)",
        "prompt": "「クラウド人工知能の進化」について解説してください。\n制約1: ちょうど3つの段落で構成すること。\n制約2: 各段落の文頭は必ず「第一に、」「第二に、」「第三に、」で始めること。\n制約3: 英語（アルファベット）を一切使用しないこと。"
    },
    {
        "type": "非同期プログラミング・アルゴリズム",
        "prompt": "Pythonで asyncio.Queue を用いた Producer-Consumer パターンの完全な動作可能コードを提示し、日本語で各コンポーネントの挙動を簡潔に解説してください。"
    },
    {
        "type": "高度ビジネス論理・ことわざ応用",
        "prompt": "「急がば回れ」ということわざの意味を説明し、大規模ソフトウェア開発のアーキテクチャ設計やリファクタリングにおいて、この教訓が極めて重要となる具体的な状況を2つ論理的に解説してください。"
    }
]
results["qualitative_ja"] = []
for jp in ja_prompts:
    full_t, content_t, reas_t, fin, usage, dt, _, _ = chat(jp["prompt"], max_tokens=1500, temperature=0.7)
    # IFEval の自動検証（最終出力 content_t に対してアルファベット混入・段落数・文頭チェック）
    eval_target = content_t if content_t else re.sub(r"<thought>.*?</thought>", "", full_t, flags=re.S).strip()
    ifeval_check = None
    if jp["type"].startswith("厳格指示追従"):
        paras = [p.strip() for p in re.split(r"\n\s*\n", eval_target.strip()) if p.strip()]
        has_3_paras = (len(paras) == 3)
        prefixes_ok = (
            len(paras) == 3
            and paras[0].startswith("第一に、")
            and paras[1].startswith("第二に、")
            and paras[2].startswith("第三に、")
        )
        latin_chars = re.findall(r"[A-Za-z]", eval_target)
        no_latin = (len(latin_chars) == 0)
        ifeval_check = {
            "paragraph_count": len(paras),
            "has_3_paragraphs": has_3_paras,
            "prefixes_matched": prefixes_ok,
            "no_english_alphabet": no_latin,
            "latin_found": latin_chars[:10],
            "all_passed": (has_3_paras and prefixes_ok and no_latin)
        }
    results["qualitative_ja"].append({
        "type": jp["type"],
        "prompt": jp["prompt"],
        "response": full_t,
        "final_content": eval_target,
        "ifeval_check": ifeval_check,
        "completion_tokens": usage.get("completion_tokens"),
        "sec": round(dt, 2)
    })
save_all()

server.terminate()
results["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
save_all()

# ================================================================
# サマリーレポート (summary.md) & UD-Q4_K_XL / 過去モデル横断比較詳細技術レポート生成
# ================================================================

# 先行実行された UD-Q4_K_XL の結果が Drive にあれば読み込んで比較テーブルを構築
q4kxl_data = {}
if os.path.exists(Q4KXL_RESULTS_PATH):
    try:
        with open(Q4KXL_RESULTS_PATH, "r", encoding="utf-8") as f:
            q4kxl_data = json.load(f)
        log(f"✅ 先行評価 UD-Q4_K_XL の結果を読み込みました: {Q4KXL_RESULTS_PATH}")
    except Exception as e:
        log(f"[!] UD-Q4_K_XL 結果の読み込みスキップ: {e}")

xl_ppl = q4kxl_data.get("perplexity_wikitext2", {}).get("ppl", "-")
xl_mmlu = f"{q4kxl_data['mmlu']['accuracy']*100:.2f}%" if "mmlu" in q4kxl_data else "-"
xl_arc = f"{q4kxl_data['arc_challenge']['accuracy']*100:.2f}%" if "arc_challenge" in q4kxl_data else "-"
xl_gsm = f"{q4kxl_data['gsm8k']['accuracy']*100:.2f}%" if "gsm8k" in q4kxl_data else "-"
xl_dspark_off = q4kxl_data.get("dspark_benchmark", {}).get("avg_tok_per_sec_off", "-")
xl_dspark_on = q4kxl_data.get("dspark_benchmark", {}).get("avg_tok_per_sec_on", "-")
xl_dspark_ratio = q4kxl_data.get("dspark_benchmark", {}).get("speedup_ratio", "-")
xl_p512 = "-"
xl_g128 = "-"
if isinstance(q4kxl_data.get("throughput"), list):
    for b in q4kxl_data["throughput"]:
        if "Prompt 512" in b["test"]:
            xl_p512 = str(b["tokens_per_sec"])
        if "Generation 128" in b["test"]:
            xl_g128 = str(b["tokens_per_sec"])

gsm_acc = results.get("gsm8k", {}).get("accuracy", 0) * 100
mmlu_acc = results.get("mmlu", {}).get("accuracy", 0) * 100
arc_acc = results.get("arc_challenge", {}).get("accuracy", 0) * 100
ppl_val = results.get("perplexity_wikitext2", {}).get("ppl", "N/A")
tc_pass = sum(1 for t in results.get("agentic_tool_calling", []) if t["success"])
tc_total = len(results.get("agentic_tool_calling", []))
dspark_off = results.get("dspark_benchmark", {}).get("avg_tok_per_sec_off", "N/A")
dspark_on = results.get("dspark_benchmark", {}).get("avg_tok_per_sec_on", "N/A")
dspark_ratio = results.get("dspark_benchmark", {}).get("speedup_ratio", "N/A")
v_pass = results.get("vision_eval", {}).get("passed_checks", 0)
v_total = results.get("vision_eval", {}).get("total_checks", 4)

summary_md = f"""# Unsloth Inkling-2-Mini GGUF 総合評価サマリー

- **対象モデル:** `unsloth/Inkling-2-Mini-GGUF` (ベース: `thinkingmachines/Inkling-2-Mini`)
- **アーキテクチャ:** Looped Transformer MoE (24層 / 中間12層×3ループ=48層適用, 総26.2B / 活性3.4B / 実効活性5.4B)
- **量子化形式:** {QUANT_NAME} + `mmproj-F16.gguf` + `dspark-Inkling-2-Mini-Q8_0.gguf`
- **評価環境:** {results.get('gpu', 'Unknown')}
- **評価期間:** {results.get('started')} 〜 {results.get('finished')}

---

## 1. 推論スループット (llama-bench)
| テスト種別 | 速度 (tokens/sec) | 標準偏差 |
| :--- | :--- | :--- |
"""
if isinstance(results.get("throughput"), list):
    for b in results["throughput"]:
        summary_md += f"| {b['test']} | {b['tokens_per_sec']} | {b['stddev']} |\n"

summary_md += f"""
## 2. DSpark 投機的デコーディング (Speculative Decoding) 加速効果
- **DSpark 無効時 (平均):** {dspark_off} tokens/sec
- **DSpark 有効時 (`draft-dspark`, n-max=3):** **{dspark_on} tokens/sec** (**{dspark_ratio}x 高速化**)

## 3. 言語モデル Perplexity (WikiText-2)
- **Perplexity:** **{ppl_val}** (ctx: 2048, 40 chunks, stderr: {results.get('perplexity_wikitext2', {}).get('stderr', 'N/A')})

## 4. 定量ベンチマーク正答率 (計 585問, SEED=42)
| ベンチマーク | 評価サンプル数 | 正答率 (Accuracy) | 平均トークン数 | 実行時間 (秒) |
| :--- | :--- | :--- | :--- | :--- |
| **MMLU** (57分野総合知識 0-shot) | {results.get('mmlu', {}).get('n', '-')} | **{mmlu_acc:.2f}%** | {results.get('mmlu', {}).get('avg_tokens', '-')} | {results.get('mmlu', {}).get('elapsed_sec', '-')}s |
| **ARC-Challenge** (高度科学推論 0-shot) | {results.get('arc_challenge', {}).get('n', '-')} | **{arc_acc:.2f}%** | {results.get('arc_challenge', {}).get('avg_tokens', '-')} | {results.get('arc_challenge', {}).get('elapsed_sec', '-')}s |
| **GSM8K** (多段階数学推論 CoT) | {results.get('gsm8k', {}).get('n', '-')} | **{gsm_acc:.2f}%** | {results.get('gsm8k', {}).get('avg_tokens', '-')} | {results.get('gsm8k', {}).get('elapsed_sec', '-')}s |

## 5. マルチモーダル Vision & エージェント Tool Calling & トークン効率
- **Vision チャート読解・視覚推論 (`mmproj-F16`):** **{v_pass} / {v_total}** チェック合格 ({results.get('vision_eval', {}).get('elapsed_sec', '-')}s)
- **Tool Calling 成功率:** **{tc_pass} / {tc_total}**
"""
for t in results.get("token_efficiency", []):
    summary_md += f"- **思考トークン消費効率 (問{t['id']}):** {t['tokens']} tokens ({t['sec']}s, {t['tok_per_sec']} tok/s)\n"

summary_md += "\n## 6. 日本語定性評価サンプル\n"
for s in results.get("qualitative_ja", []):
    ifeval_badge = ""
    if s.get("ifeval_check"):
        ic = s["ifeval_check"]
        ifeval_badge = f"\n- **IFEval 自動判定:** {'✅ 完全合格' if ic['all_passed'] else '⚠️ 一部違反'} (3段落={ic['has_3_paragraphs']}, 文頭一致={ic['prefixes_matched']}, 英字なし={ic['no_english_alphabet']})\n"
    summary_md += f"### 【{s['type']}】{ifeval_badge}\n**Q:** {s['prompt']}\n\n**A:**\n{s['response']}\n\n*(トークン数: {s['completion_tokens']}, 所要時間: {s['sec']}秒)*\n\n---\n"

for d in [LOCAL_OUT, DRIVE_OUT]:
    with open(os.path.join(d, "summary.md"), "w", encoding="utf-8") as f:
        f.write(summary_md)

# 詳細評価技術レポート (過去3モデルとの横断比較付き)
gen128_tps = "-"
prompt512_tps = "-"
if isinstance(results.get("throughput"), list):
    for b in results["throughput"]:
        if "Prompt 512" in b["test"]:
            prompt512_tps = str(b["tokens_per_sec"])
        if "Generation 128" in b["test"]:
            gen128_tps = str(b["tokens_per_sec"])

detailed_report_md = f"""# Unsloth Inkling-2-Mini ({QUANT_NAME}) 詳細評価技術レポート（過去モデル横断比較付き）

**作成日**: {time.strftime('%Y年%m月%d日')}
**評価対象モデル**: [unsloth/Inkling-2-Mini-GGUF](https://huggingface.co/unsloth/Inkling-2-Mini-GGUF) (Base: `thinkingmachines/Inkling-2-Mini`)
**アーキテクチャ**: Multimodal Autoregressive Looped Transformer MoE (24層デコーダ / 中間12層を3回ループ=48層適用, 128 Experts 中 Top-8 ルーティング, 総パラメータ 26.2B / 活性 3.4B / 実効活性 5.4B)
**量子化・付属品**: `{sel['main_gguf']}` + Vision Projector (`mmproj-F16.gguf`) + DSpark Drafter (`dspark-Inkling-2-Mini-Q8_0.gguf`)
**検証インフラ**: Google Colab ({results.get('gpu', 'NVIDIA GPU')})
**推論エンジン**: Unsloth `llama.cpp` (`b12588-mix-c0c3de8`, `inkling2_loop` & `draft-dspark` & per-expert NVFP4 scales 対応ビルド)
**保存先ディレクトリ**: `Google Drive/MyDrive/unsloth-inkling-2-mini-q4km-eval-results/`

---

## 1. エグゼクティブサマリー

本レポートは、Thinking Machines Lab 初のオープンウェイト Looped Transformer モデル **Inkling-2-Mini** の Unsloth Dynamic 3.0 量子化版（`unsloth/Inkling-2-Mini-GGUF`）に対し、これまでの Colab 評価シリーズ（**Gemma-4-31B-it Q4_K_M**、**Qwen3.8-27B UD-Q4_K_XL**、**Gemma-4-E4B-it Q4_K_M**）と完全に同一のベンチマーク母集団（SEED=42）を用いて実施した総合性能評価レポートです。

### 1.1 過去 Colab 評価モデルとの横断比較テーブル

| 評価項目 | Gemma-4-31B (Q4_K_M) | Qwen3.8-27B (UD-Q4_K_XL) | Inkling-2-Mini (UD-Q4_K_XL) | **Inkling-2-Mini (UD-Q4_K_M)** |
| :--- | :---: | :---: | :---: | :---: |
| **アーキテクチャ** | Dense 31B | Dense 27B | Looped MoE 26.2B (活性5.4B) | **Looped MoE 26.2B (活性5.4B)** |
| **モデルサイズ (GGUF)** | 18.8 GB | 16.5 GB | 17.63 GB (+1.04GB) | **16.61 GB (+1.04GB / -1.02GB軽量)** |
| **検証 GPU** | NVIDIA L4 (24GB) | NVIDIA L4 (24GB) | {q4kxl_data.get('gpu', 'NVIDIA L4')} | **{results.get('gpu', 'NVIDIA GPU')}** |
| **Prompt 速度 (512 tok)** | 739.16 tok/s | 882.58 tok/s | {xl_p512} tok/s | **{prompt512_tps} tok/s** |
| **生成速度 (128 tok / DSpark ON)** | 12.72 tok/s | 14.02 tok/s | {xl_g128} tok/s (DSpark: {xl_dspark_on}) | **{gen128_tps} tok/s (DSpark: {dspark_on} tok/s)** |
| **WikiText-2 Perplexity** | 6929.09 | 5.9678 | {xl_ppl} | **{ppl_val}** |
| **MMLU (285問, 0-shot)** | 12.63% | 50.88% | {xl_mmlu} | **{mmlu_acc:.2f}%** |
| **ARC-Challenge (200問)** | 16.50% | 80.50% | {xl_arc} | **{arc_acc:.2f}%** |
| **GSM8K (100問, CoT)** | 83.00% | 93.00% | {xl_gsm} | **{gsm_acc:.2f}%** |
| **Tool Calling 成功率** | - | 2 / 2 (100%) | 2 / 2 (100%) | **{tc_pass} / {tc_total} ({tc_pass/max(1,tc_total)*100:.0f}%)** |
| **Vision マルチモーダル** | 非対応 (Textのみ検証) | 非対応 (Textのみ検証) | 非対応 (Textのみ検証) | **{v_pass} / {v_total} 合格 (`mmproj-F16`)** |

---

## 2. アーキテクチャ特性と推論スループット・DSpark 分析

### 2.1 Looped Transformer × MoE の計算特性
Inkling-2-Mini は全24層のデコーダ層のうち、中間12層（Layer 6〜17）を重み共有のまま **3回ループ適用** することで、24層分のメモリフットプリント（4-bit で約 17.6GB）でありながら **48層相当の深層表現力** を獲得しています。1パスあたりの活性パラメータは 3.4B ですが、中間層3ループによりトークンあたり **実効 5.4B パラメータ** が活性化します。

### 2.2 llama-bench スループット計測結果
```text
"""
if isinstance(results.get("throughput"), list):
    for b in results["throughput"]:
        detailed_report_md += f"- {b['test']}: {b['tokens_per_sec']} ± {b['stddev']} tokens/sec\n"

detailed_report_md += f"""```

### 2.3 DSpark 投機的デコーディング (`dspark-Inkling-2-Mini-Q8_0.gguf`) の実測効果
402MB の専用ドラフトモデル `dspark-Inkling-2-Mini-Q8_0.gguf` を用いた投機的デコーディング（`--spec-type draft-dspark --spec-draft-n-max 3`）の有無による実効生成速度の比較結果は以下の通りです：

| タスク | DSpark OFF (tok/s) | DSpark ON (tok/s) | 高速化倍率 |
| :--- | :---: | :---: | :---: |
"""
off_list = results.get("dspark_benchmark", {}).get("without_dspark", [])
on_list = results.get("dspark_benchmark", {}).get("with_dspark", [])
for o, w in zip(off_list, on_list):
    ratio = round(w["tokens_per_sec"] / max(o["tokens_per_sec"], 0.01), 2)
    detailed_report_md += f"| {o['task']} | {o['tokens_per_sec']} tok/s | **{w['tokens_per_sec']} tok/s** | **{ratio}x** |\n"

detailed_report_md += f"""| **平均 (Average)** | **{dspark_off} tok/s** | **{dspark_on} tok/s** | **{dspark_ratio}x** |

---

## 3. マルチモーダル Vision 評価 (`mmproj-F16.gguf`)

2026年四半期売上バーチャート画像（PNG）を入力し、数値の正確な読み取り・合計計算・成長率比較を行わせた結果：
- **合格チェック数**: **{v_pass} / {v_total}**
- **詳細判定**: `{json.dumps(results.get('vision_eval', {}).get('checks', {}), ensure_ascii=False)}`
- **モデル出力抜粋**:
```text
{results.get('vision_eval', {}).get('response', 'N/A')[:1500]}
```

---

## 4. エージェント Tool Calling と思考トークン効率

### 4.1 Tool Calling 検証結果 ({tc_pass}/{tc_total})
```json
{json.dumps(results.get('agentic_tool_calling', []), indent=2, ensure_ascii=False)}
```

### 4.2 思考トークン消費効率 (Token Efficiency)
"""
for t in results.get("token_efficiency", []):
    detailed_report_md += f"- **論理パズル問{t['id']}**: {t['tokens']} tokens ({t['sec']}s, {t['tok_per_sec']} tok/s)\n"

detailed_report_md += """
---

## 5. 日本語定性評価 & IFEval 厳格指示追従
"""
for s in results.get("qualitative_ja", []):
    detailed_report_md += f"### 5.{results['qualitative_ja'].index(s)+1} {s['type']}\n"
    if s.get("ifeval_check"):
        detailed_report_md += f"- **自動検証結果**: `{json.dumps(s['ifeval_check'], ensure_ascii=False)}`\n"
    detailed_report_md += f"**プロンプト**: {s['prompt']}\n\n**回答**:\n{s['response']}\n\n---\n"

for d in [LOCAL_OUT, DRIVE_OUT]:
    with open(os.path.join(d, "detailed_evaluation_report.md"), "w", encoding="utf-8") as f:
        f.write(detailed_report_md)

log(f"✅ 全成果物 (summary.md, detailed_evaluation_report.md, 各種jsonl, results.json) を Google Drive に保存完了: {DRIVE_OUT}")
print(summary_md)

=== llama.cpp 認識デバイス一覧 ===
Available devices:
  CUDA0: NVIDIA L4 (22563 MiB, 22369 MiB free)
[01:24:48] Target GPU: NVIDIA L4, 23034 MiB, 580.82.07 | Model: /content/models/Inkling-2-Mini-UD-Q4_K_M.gguf
[01:24:48] === [1/9] llama-bench (推論スループット計測) ===
[01:26:03] Throughput: [{'test': 'Prompt 512 tok', 'tokens_per_sec': 1076.05, 'stddev': 10.02}, {'test': 'Prompt 1024 tok', 'tokens_per_sec': 1045.16, 'stddev': 4.19}, {'test': 'Prompt 2048 tok', 'tokens_per_sec': 911.53, 'stddev': 3.68}, {'test': 'Prompt 4096 tok', 'tokens_per_sec': 783.54, 'stddev': 3.15}, {'test': 'Generation 128 tok', 'tokens_per_sec': 36.55, 'stddev': 0.21}, {'test': 'Generation 256 tok', 'tokens_per_sec': 36.52, 'stddev': 0.08}]
[01:26:03] === [2/9] llama-perplexity (WikiText-2 言語モデリング精度) ===


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

... (36 outputs left out)

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

[02:45:35] 実行中: gsm8k (計 100 問, max_tokens=1024)
[03:04:28] ✅ gsm8k 完了: 正答率 91.00% (100問, 1132.5秒, 切断:9件)
[03:04:28] === [7/9] マルチモーダル Vision 評価 (チャート解析・視覚推論) ===
[03:04:28] [!] Vision 評価エラー: HTTP Error 500: Internal Server Error
[03:04:28] === [8/9] エージェント Tool Calling & Looped Transformer 思考トークン効率テスト ===
[03:06:17] === [9/9] 日本語定性評価 & 厳格指示追従 (IFEval 三重制約) ===
[03:08:29] ✅ 先行評価 UD-Q4_K_XL の結果を読み込みました: /content/drive/MyDrive/unsloth-inkling-2-mini-eval-results/results.json
[03:08:29] ✅ 全成果物 (summary.md, detailed_evaluation_report.md, 各種jsonl, results.json) を Google Drive に保存完了: /content/drive/MyDrive/unsloth-inkling-2-mini-q4km-eval-results
# Unsloth Inkling-2-Mini GGUF 総合評価サマリー

- **対象モデル:** `unsloth/Inkling-2-Mini-GGUF` (ベース: `thinkingmachines/Inkling-2-Mini`)
- **アーキテクチャ:** Looped Transformer MoE (24層 / 中間12層×3ループ=48層適用, 総26.2B / 活性3.4B / 実効活性5.4B)
- **量子化形式:** UD-Q4_K_M (Dynamic 3.0, 16.61GB) + `mmproj-F16.gguf` + `dspark-Inkling-2-Mini-Q8_0.gguf`
- **評価環境:** NVIDIA L4, 23034 MiB, 

## 6. Colab の自動切断・VM 解放
評価結果が Google Drive に安全に書き込まれた後、不要なコンピュートユニット消費を防ぐために Colab ランタイムの接続を自動的に解除します。

In [6]:
import time
from google.colab import runtime

print("評価結果の Google Drive への書き込みがすべて完了しました。")
print("15秒後に Colab ランタイムを切断し、リソースを解放します...")
time.sleep(15)

# ランタイムの解放 (自動切断)
runtime.unassign()

評価結果の Google Drive への書き込みがすべて完了しました。
15秒後に Colab ランタイムを切断し、リソースを解放します...
